# Text-to-Speech (TTS) Reader (`window.craftTTS`)

> **Feature 4.2:** Zero-setup Web Speech API (`window.speechSynthesis`) reader for prose notes and AI responses. Cleans Markdown/URLs/hex/paths, replaces code blocks and tables with concise spoken placeholders, chunks by sentence to avoid Chrome's long-utterance stall bug, highlights the active cell, and provides a compact floating player bar.

## Templates (`#| export_ft`)

In [ ]:
#| export_ft CRAFT.js
#| eval: false
TEMPLATES = {
    'tts_bar': Div(
        Button('⏮', type='button', cls='craft-tts-btn', aria_label='Previous cell', title='Previous cell', data_ref='prev'),
        Button('⏸', type='button', cls='craft-tts-btn', aria_label='Play or pause', title='Play / Pause', data_ref='play'),
        Button('⏭', type='button', cls='craft-tts-btn', aria_label='Next cell', title='Next cell', data_ref='next'),
        Select(
            Option('1.0×', value='1'),
            Option('1.25×', value='1.25'),
            Option('1.5×', value='1.5'),
            Option('1.75×', value='1.75'),
            Option('2.0×', value='2'),
            cls='uk-select craft-tts-rate', aria_label='Reading speed', data_ref='rate',
        ),
        Span('', cls='craft-tts-status', data_ref='status'),
        Button('×', type='button', cls='craft-tts-btn', aria_label='Stop reading', title='Stop', data_ref='stop'),
        id='craft-tts-bar', role='region', aria_label='Text-to-speech player',
    ),
}

## TTS Reader Runtime (`#| export_js_iife`)

In [ ]:
#| export_js_iife CRAFT.js craftTTS
#| eval: false
window.craftTTS?.disable?.();
if (!document.getElementById('dialog-container')) return;

const CORE = window.__craftCMCore;
const {qsa, on, store, mountTpl, notifyErr, normId, getCellEl, allCells, primaryCell, cellMtype} = CORE;
const ac = new AbortController(), sig = {signal: ac.signal};
const RATE_KEY = 'craft-tts-rate';

/* Clean rendered DOM or text into speakable prose */
function extractSpeakableText(cell, preferOutput = false) {
  if (!cell) return '';
  const mtype = cellMtype(cell);
  if (mtype === 'code' && !preferOutput) {
    const code = cell.querySelector(':scope > .card-in .msg-content pre')?.textContent || '';
    const lines = code.trim() ? code.trim().split('\n').length : 0;
    return lines ? `Python code block, ${lines} ${lines === 1 ? 'line' : 'lines'}.` : '';
  }
  const targetCard = (preferOutput && cell.querySelector(':scope > .card-out .msg-content')) ||
                     (mtype === 'prompt' && cell.querySelector(':scope > .card-out .msg-content')) ||
                     cell.querySelector(':scope > .card-in .msg-content');
  if (!targetCard) return '';
  const clone = targetCard.cloneNode(true);
  for (const pre of qsa('pre', clone)) {
    const cls = pre.querySelector('code')?.className || '';
    const langM = cls.match(/language-(\w+)/);
    const lang = langM ? langM[1] : 'Code';
    const lines = (pre.textContent || '').trim().split('\n').length;
    pre.replaceWith(document.createTextNode(` ${lang} code block, ${lines} ${lines === 1 ? 'line' : 'lines'}. `));
  }
  for (const tbl of qsa('table', clone)) {
    const cols = tbl.querySelector('tr')?.children?.length || 0;
    tbl.replaceWith(document.createTextNode(` Table, ${cols} ${cols === 1 ? 'column' : 'columns'}. `));
  }
  let text = (clone.textContent || '').replace(/\s+/g, ' ').trim();
  // Strip bare URLs, inline hex hashes, and long filesystem paths
  text = text
    .replace(/https?:\/\/\S+/g, 'link')
    .replace(/\b0x[0-9a-f]{6,}\b/gi, '')
    .replace(/(?:\/[\w.-]+){3,}/g, 'file path')
    .replace(/[#*_`~>\[\]]+/g, ' ')
    .replace(/\s+/g, ' ')
    .trim();
  return text;
}

/* Chunk long text by sentence to avoid Chrome's ~15s speechSynthesis stall bug */
function splitLongSegment(seg, maxLen) {
  const s = seg.trim();
  if (!s) return [];
  if (s.length <= maxLen) return [s];
  const out = [];
  let cur = '';
  for (const part of s.split(/(?<=[,;:])\s+|\s+/)) {
    if (!part) continue;
    if (part.length > maxLen) {
      if (cur) { out.push(cur); cur = ''; }
      for (let i = 0; i < part.length; i += maxLen) out.push(part.slice(i, i + maxLen));
    } else if (cur && cur.length + 1 + part.length > maxLen) {
      out.push(cur);
      cur = part;
    } else {
      cur = cur ? `${cur} ${part}` : part;
    }
  }
  if (cur) out.push(cur);
  return out;
}

const ABBREV_RE = /\b(?:e\.g|i\.e|dr|mr|mrs|ms|prof|sr|jr|vs|etc|fig|no|st)\.$/i;

function chunkSentences(text, maxLen = 220) {
  const raw = String(text || '').replace(/\s+/g, ' ').trim();
  if (!raw) return [];
  const sentences = [];
  let buf = '';
  for (const tok of raw.split(' ')) {
    buf = buf ? `${buf} ${tok}` : tok;
    if (/[.!?]+['")\]]*$/.test(tok) && !ABBREV_RE.test(tok) && !/\d+\.\d+/.test(tok)) {
      sentences.push(buf);
      buf = '';
    }
  }
  if (buf) sentences.push(buf);
  const chunks = [];
  let cur = '';
  for (const sent of sentences) {
    for (const piece of splitLongSegment(sent, maxLen)) {
      if (cur && cur.length + 1 + piece.length > maxLen) {
        chunks.push(cur);
        cur = piece;
      } else {
        cur = cur ? `${cur} ${piece}` : piece;
      }
    }
  }
  if (cur) chunks.push(cur);
  return chunks;
}


### Speech Queue, Player Bar & Playback Controls

In [ ]:
#| export_js_iife CRAFT.js craftTTS
#| eval: false
/* Speech Queue, Player Bar & Playback Controls */
let bar = null, queue = [], cellIdx = 0, chunkIdx = 0, chunks = [], playing = false, paused = false;
let activeCellId = null;
let rate = parseFloat(store.get(RATE_KEY, '1')) || 1;

function clearHighlight() {
  if (activeCellId) getCellEl(activeCellId)?.classList.remove('craft-tts-active');
  qsa('#dialog-container > .editable.craft-tts-active').forEach(c => c.classList.remove('craft-tts-active'));
  activeCellId = null;
}

function syncBar() {
  if (!playing && !paused) { bar?.root?.remove(); clearHighlight(); return; }
  if (!bar) {
    bar = mountTpl('tts_bar', {refs: true});
    bar.rate.value = String(rate);
    on(bar.play, 'click', () => togglePause(), sig);
    on(bar.prev, 'click', () => jumpCell(-1), sig);
    on(bar.next, 'click', () => jumpCell(1), sig);
    on(bar.stop, 'click', () => stop(), sig);
    on(bar.rate, 'change', () => {
      rate = parseFloat(bar.rate.value) || 1;
      store.set(RATE_KEY, String(rate));
      if (playing && !paused) speakCurrentChunk();
    }, sig);
  }
  if (!bar.root.isConnected) document.body.append(bar.root);
  bar.play.textContent = paused ? '▶' : '⏸';
  bar.status.textContent = `Cell ${cellIdx + 1}/${queue.length}`;
}

function speakCurrentChunk() {
  const synth = window.speechSynthesis;
  if (!synth) return notifyErr('Web Speech API is not supported in this browser.', 'craftTTS');
  synth.cancel();
  while (cellIdx < queue.length && !getCellEl(queue[cellIdx].id)) {
    cellIdx++; chunks = []; chunkIdx = 0;
  }
  if (cellIdx >= queue.length) { stop(); return; }
  const {id, preferOutput} = queue[cellIdx];
  const cell = getCellEl(id);
  clearHighlight();
  activeCellId = id;
  cell.classList.add('craft-tts-active');
  cell.scrollIntoView({behavior: 'smooth', block: 'nearest'});
  if (!chunks.length) {
    chunks = chunkSentences(extractSpeakableText(cell, preferOutput));
    chunkIdx = 0;
  }
  if (chunkIdx >= chunks.length) {
    cellIdx++; chunks = []; chunkIdx = 0;
    if (cellIdx < queue.length) speakCurrentChunk();
    else stop();
    return;
  }
  const utt = new SpeechSynthesisUtterance(chunks[chunkIdx]);
  utt.rate = rate;
  utt.onend = () => {
    if (!playing || paused) return;
    chunkIdx++;
    speakCurrentChunk();
  };
  utt.onerror = e => {
    if (e.error === 'interrupted' || e.error === 'canceled') return;
    console.warn('craftTTS utterance error', e.error);
    chunkIdx++;
    if (playing && !paused) speakCurrentChunk();
  };
  syncBar();
  synth.speak(utt);
}

function readCells(cells, preferOutput = false) {
  if (!window.speechSynthesis) return notifyErr('Text-to-speech is not available in this browser.', 'craftTTS');
  const list = cells.filter(Boolean).map(c => ({id: normId(c.id), preferOutput})).filter(item => item.id);
  if (!list.length) return notifyErr('No readable cell selected.', 'craftTTS');
  queue = list; cellIdx = 0; chunks = []; chunkIdx = 0; playing = true; paused = false;
  // Initiate speech synchronously inside the user-gesture handler (required for mobile Safari)
  speakCurrentChunk();
}

function readCell(cell = primaryCell(), preferOutput = false) {
  if (cell) readCells([cell], preferOutput);
}

function readFromHere(cell = primaryCell()) {
  const all = allCells({visible: true});
  const idx = cell ? all.indexOf(cell) : -1;
  if (idx < 0) return notifyErr('Select a cell first to read from here.', 'craftTTS');
  readCells(all.slice(idx), false);
}

function togglePause() {
  const synth = window.speechSynthesis;
  if (!synth || !playing) return;
  if (paused) { paused = false; synth.resume(); }
  else { paused = true; synth.pause(); }
  syncBar();
}

function jumpCell(delta) {
  if (!playing) return;
  cellIdx = Math.max(0, Math.min(queue.length - 1, cellIdx + delta));
  chunks = []; chunkIdx = 0; paused = false;
  speakCurrentChunk();
}

function stop() {
  playing = false; paused = false; queue = []; chunks = [];
  try { window.speechSynthesis?.cancel(); } catch {}
  syncBar();
}

### Lifecycle, Action Menu & Hotkeys

In [ ]:
#| export_js_iife CRAFT.js craftTTS
#| eval: false
/* Lifecycle, Action Menu & Hotkeys */
on(window, 'beforeunload', stop, sig);
on(window, 'pagehide', stop, sig);

const offMenu = window.craftActionMenus?.registerProvider?.(ctx => {
  if (!ctx.cell) return [];
  return [
    {header: 'Read Aloud'},
    {label: 'Read aloud', onClick: () => readCell(ctx.cell, ctx.isOutput)},
    {label: 'Read from here', onClick: () => readFromHere(ctx.cell)},
  ];
});

const reg = spec => window.__hotkeyRegister?.(spec);
reg({id: 'craft-tts-cell', label: 'Read selected cell aloud (TTS)', group: 'CRAFT · Reading', combo: '', handler: () => { readCell(); return true; }});
reg({id: 'craft-tts-from', label: 'Read dialog from selected cell (TTS)', group: 'CRAFT · Reading', combo: '', handler: () => { readFromHere(); return true; }});
reg({id: 'craft-tts-stop', label: 'Stop text-to-speech reader', group: 'CRAFT · Reading', combo: '', handler: () => { stop(); return true; }});

window.craftTTS = {
  readCell, readFromHere, togglePause, stop, extractSpeakableText, chunkSentences,
  disable() {
    stop();
    ac.abort(); offMenu?.();
    bar?.root?.remove();
    delete window.craftTTS;
  },
};

## TTS Reader CSS (`#| export_css`)

In [ ]:
#| export_css CRAFT.css
#| eval: false
/* ── Text-to-Speech Reader (18_tts_reader) ────────────────────────── */
.editable.craft-tts-active {
  box-shadow: inset 3px 0 0 var(--craft-cyan) !important;
  background: color-mix(in srgb, var(--craft-cyan) 6%, transparent) !important;
}
#craft-tts-bar {
  position: fixed;
  right: 16px;
  bottom: calc(16px + env(safe-area-inset-bottom, 0px));
  z-index: 1230;
  display: flex;
  align-items: center;
  gap: 6px;
  padding: 6px 10px;
  border-radius: 999px;
  background: var(--craft-recessed);
  border: 1px solid var(--craft-cyan);
  box-shadow: 0 10px 28px rgba(0,0,0,.55);
  color: var(--craft-fg);
  font-size: 11.5px;
}
html.craft-mobile #craft-tts-bar {
  left: 50%; right: auto; transform: translateX(-50%);
  bottom: calc(64px + env(safe-area-inset-bottom, 0px));
}
.craft-tts-btn {
  border: 0; background: transparent; color: var(--craft-fg);
  font-size: 13px; padding: 2px 6px; border-radius: 4px; cursor: pointer;
}
.craft-tts-btn:hover { background: var(--craft-hover); color: var(--craft-fg-strong); }
.craft-tts-rate { width: auto !important; height: 24px !important; padding: 0 20px 0 6px !important; font-size: 11px !important; }
.craft-tts-status { font-family: var(--craft-code-family); font-size: 10.5px; color: var(--craft-cyan); }

Verify `18_tts_reader.ipynb` exports `window.craftTTS`, its template, and CSS.

In [ ]:
import json, os, subprocess
from craft_dev.auth import REPO_ROOT
from craft_dev.build import extract_nb_assets

assets = extract_nb_assets(REPO_ROOT/'nbs/craft/18_tts_reader.ipynb')
js = '\n'.join(code for _, code in assets['CRAFT.js'])
css = '\n'.join(code for _, code in assets['CRAFT.css'])
for sym in ('window.craftTTS', 'tts_bar', 'extractSpeakableText', 'chunkSentences', 'primaryCell'):
    assert sym in js, f"Missing {sym}"
assert '#craft-tts-bar' in css and '.craft-tts-active' in css

node_script = r"""
const src = process.env.TTS_JS;
const i0 = src.indexOf('function splitLongSegment');
const i1 = src.indexOf('/* Speech Queue');
const {chunkSentences} = new Function(src.slice(i0, i1) + '; return {chunkSentences};')();

// 1. Empty / whitespace
if (chunkSentences('   ').length !== 0) throw new Error('Expected empty');

// 2. Decimals and abbreviations must not force premature splits
const s = 'Dr. Smith measured 3.14, e.g. in Fig. 2. Next sentence!';
const c1 = chunkSentences(s, 45);
if (c1.length !== 2 || c1[0] !== 'Dr. Smith measured 3.14, e.g. in Fig. 2.' || c1[1] !== 'Next sentence!')
  throw new Error('Abbreviation/decimal split mismatch: ' + JSON.stringify(c1));

// 3. Long punctuation-free text splits on word boundaries within maxLen
const longProse = Array.from({length: 60}, (_, i) => 'word' + i).join(' ');
const c2 = chunkSentences(longProse, 80);
if (c2.length < 4 || c2.some(x => x.length > 80))
  throw new Error('Long prose chunk exceeded maxLen: ' + JSON.stringify(c2));

// 4. Single overlong token is sliced cleanly
const c3 = chunkSentences('a'.repeat(125), 50);
if (c3.length !== 3 || c3[0].length !== 50 || c3[2].length !== 25)
  throw new Error('Overlong token split mismatch: ' + JSON.stringify(c3));

process.stdout.write(JSON.stringify({ok: true, counts: [c1.length, c2.length, c3.length]}));
"""
res = subprocess.run(['node', '-e', node_script], env={**os.environ, 'TTS_JS': js}, capture_output=True, text=True, check=True)
assert json.loads(res.stdout)['ok']
